# Interactive 3D scatter explorer (scatter_cost)

Live alternative to `analyze.py`'s Plotly 3D plots. Uses `pandas` for the
median aggregation (fast enough on its own -- no out-of-core tooling needed)
and `k3d` (WebGL, actively maintained) for rendering, which handles far more
points smoothly than Plotly's inline-HTML `Scatter3d`.

(An earlier version used `ipyvolume` instead of `k3d` -- dropped after
hitting a GLSL shader compile error ["undeclared identifier"] in its
`pythreejs`-based renderer, a real bug in that unmaintained dependency
(no release since ~2021), not fixable from notebook code.)

Reads `results_agg.csv` (produced once by `doit agg`, see dodo.py) instead of
re-parsing the raw per-iteration `results.csv` and re-computing the median on
every notebook restart -- run `doit agg` first (or after regenerating
`results.csv`) if it's missing or stale.

Filtering works by toggling each point's size to 0 rather than removing it
from the arrays -- avoids resizing the GPU buffer on every slider drag, which
is what keeps this smooth even with millions of points.

Run with `jupyter lab`, or open directly in VS Code's notebook editor.


In [1]:
import pathlib
import sys

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

sys.path.insert(0, str(pathlib.Path.cwd()))
from analyze import (
    PROBLEM,
    LATENCY,
    _log_ticks,
    principal_dims,
)  # reuse Dim descriptors + log-tick helper
from matplotlib.ticker import MaxNLocator

AGG_CSV = "plots/sg/results_agg.csv"  # produced by `doit agg`; point at your sweep

agg = pd.read_csv(AGG_CSV)
print(f"{len(agg):,} points")

248,158 points


In [2]:
import k3d
import k3d.colormaps.matplotlib_color_maps as mcm

x_dim, y_dim = principal_dims(agg)
z_dim = LATENCY
color_dim = PROBLEM.group

# k3d renders positions as literal world-space coordinates with a single
# shared scale across all 3 axes -- unlike Plotly's 3D scene, it can't give
# each axis its own independent domain. block_size/latency's *log*-space
# range (a few units) is naturally much narrower than blocks_per_dpu's raw
# range (1-24), so without rescaling, latency/block_size look squashed flat.
# Fix: independently normalize each axis's (possibly log-transformed) values
# to the same [0, AXIS_SPAN] span, then place real-value tick labels (not
# k3d's own, which would just show the meaningless normalized coordinate) at
# the corresponding normalized position.
AXIS_SPAN = 10.0


def to_axis_space(dim, real_values):
    v = np.asarray(real_values, dtype=float)
    if dim.log == 2:
        return np.log2(v)
    if dim.log == 10:
        return np.log10(v)
    return v


def axis_label(dim):
    def escape(s):
        return s.replace("_", "\\_")

    return (
        f"\\operatorname{{log}}_{{{dim.log}}}(\\mathrm{{{escape(str(dim.col))}}})"
        if dim.log
        else f"\\mathrm{{{escape(dim.col)}}}"
    )


def nice_ticks(dim, real_values):
    # Real-valued tick marks for `dim`: nice log-spaced powers for a
    # log-scaled dim (matching analyze.py's own colorbar/axis ticks), nice
    # round linear numbers otherwise.
    if dim.log is not None:
        return _log_ticks(real_values, dim.log)
    lo, hi = float(np.min(real_values)), float(np.max(real_values))
    return [t for t in MaxNLocator(nbins=6).tick_values(lo, hi) if lo <= t <= hi]


class Axis:
    # One plotted dimension: its own (lo, hi) in log-space (if any) and the
    # normalize() mapping real values -> this axis's shared [0, AXIS_SPAN]
    # world-space coordinate.
    def __init__(self, dim, real_values, span=AXIS_SPAN):
        self.dim = dim
        self.span = span
        space = to_axis_space(dim, real_values)
        self.lo, self.hi = float(space.min()), float(space.max())

    def normalize(self, real_values):
        space = to_axis_space(self.dim, real_values)
        return (space - self.lo) / (self.hi - self.lo) * self.span


x_axis = Axis(x_dim, agg[x_dim.col])
y_axis = Axis(y_dim, agg[y_dim.col])
z_axis = Axis(z_dim, agg[z_dim.col], span=AXIS_SPAN / 2)

x, y, z = (
    x_axis.normalize(agg[x_dim.col]),
    y_axis.normalize(agg[y_dim.col]),
    z_axis.normalize(agg[z_dim.col]),
)
positions = np.column_stack([x, y, z]).astype(np.float32)
attr = to_axis_space(color_dim, agg[color_dim.col]).astype(np.float32)

BASE_SIZE = 0.05

points = k3d.points(
    positions=positions,
    attribute=attr,
    color_map=mcm.Viridis,
    color_range=[float(attr.min()), float(attr.max())],
    point_sizes=np.full(len(agg), BASE_SIZE, dtype=np.float32),
    # "flat" is a cheap camera-facing billboard shader -- handles far more
    # points smoothly than the default "3dSpecular" (lit 3D mesh per point).
    shader="flat",
)

plot = k3d.plot()
plot.axes = [axis_label(x_dim), axis_label(y_dim), axis_label(z_dim)]
plot += points

# Every axis now spans the same [0, AXIS_SPAN] world-space range, so 0 is
# every axis's own low edge -- place each dim's own tick labels along the
# edge where the other two axes sit at their low edge.
for label in [
    k3d.text(
        f"{real_val:g}",
        position=[x_axis.normalize([real_val])[0], 0, 0],
        color=0,
        size=0.8,
    )
    for real_val in nice_ticks(x_dim, agg[x_dim.col])
]:
    plot += label
for label in [
    k3d.text(
        f"{real_val:g}",
        position=[0, y_axis.normalize([real_val])[0], 0],
        color=0,
        size=0.8,
    )
    for real_val in nice_ticks(y_dim, agg[y_dim.col])
]:
    plot += label
for label in [
    k3d.text(
        f"{real_val:g}",
        position=[0, 0, z_axis.normalize([real_val])[0]],
        color=0,
        size=0.8,
    )
    for real_val in nice_ticks(z_dim, agg[z_dim.col])
]:
    plot += label


def make_slider(dim):
    values = sorted(int(v) for v in agg[dim.col].unique())
    return widgets.SelectionRangeSlider(
        options=values,
        index=(0, len(values) - 1),
        description=dim.col,
        continuous_update=True,
        layout=widgets.Layout(width="500px"),
    )


sliders = {d.col: make_slider(d) for d in PROBLEM.all_dims}


def update(*_):
    mask = np.ones(len(agg), dtype=bool)
    for d in PROBLEM.all_dims:
        lo, hi = sliders[d.col].value
        col = agg[d.col].to_numpy()
        mask &= (col >= lo) & (col <= hi)
    points.point_sizes = np.where(mask, BASE_SIZE, 0.0).astype(np.float32)


for s in sliders.values():
    s.observe(update, names="value")

display(widgets.VBox(list(sliders.values())))
plot.display()

Output()